# Projeto Terra Signal - Telecomunicações

## Entendimento do problema

A Terra Signal é uma empresa fictícia de telecomunicações. A base `history.csv`, disponibilizada pela Databricks, traz a carteira histórica de clientes: características de cada cliente, serviços contratados, tipo de contrato, informações financeiras e se o cliente cancelou ou não.

**Problema (definido pelo projeto):** como estruturar, processar, armazenar e disponibilizar de forma segura os dados da Terra Signal, para que possam ser utilizados em análises capazes de apoiar a compreensão de sua carteira de clientes?

**Objetivo geral:** desenvolver uma solução integrada para estruturar, processar, armazenar e analisar os dados da Terra Signal, utilizando a Arquitetura Medalhão e técnicas de análise exploratória, de modo a produzir informações confiáveis que apoiem a compreensão de sua carteira de clientes.

## Objetivo específico do grupo

Dentro desse objetivo geral, escolhemos um foco: **entender o cancelamento (churn)**. Mais de 1 em cada 4 clientes da base cancelou, e é aí que a compreensão da carteira tem mais valor prático para a empresa.

**Pergunta central:** quais características da carteira de clientes estão associadas ao cancelamento, e quais grupos concentram esse risco?

Ela foi dividida em três frentes, uma para cada camada Gold:

| Gold | Frente | Pergunta |
|---|---|---|
| 1 | Cliente x Financeiro | O perfil e a situação financeira explicam quanto o cliente paga e se ele cancela? |
| 2 | Cliente x Serviços | Quais serviços cada perfil contrata e quais combinações seguram ou afastam o cliente? |
| 3 | Cliente x Contrato | Como contrato, pagamento e tempo de permanência se relacionam com o cancelamento? |

O resultado esperado é uma lista curta de fatores e grupos de risco que a Terra Signal possa usar para decidir onde agir primeiro.

## Etapas do projeto

| Etapa | Notebook | O que faz |
|---|---|---|
| Bronze | `camada_bronze.ipynb` | Recebe o arquivo original sem alterar nada e avalia a qualidade dos dados |
| Silver | `camada_silver.ipynb` | Corrige os problemas encontrados na Bronze e organiza a base |
| Estatística | `camada_estatística.ipynb` | Descreve cada variável e as primeiras relações com o cancelamento |
| Gold 1, 2 e 3 | `camada_gold*.ipynb` | Tabelas preparadas por tema e respostas às perguntas |
| Banco de dados | `banco_de_dados.ipynb` | Guarda todas as camadas em SQL e responde as perguntas principais com consultas |

Segurança da informação é tratada ao longo do caminho: verificação de integridade do arquivo na Bronze, separação entre dados identificáveis (Silver) e dados para análise (Gold), pseudonimização do ID do cliente e controle de acesso por visões no banco.

## Variáveis da base

- **Cliente:** `customerID` (ID), `gender` (gênero), `SeniorCitizen` (idoso), `Partner` (tem cônjuge), `Dependents` (tem dependentes)
- **Relacionamento:** `tenure` (meses como cliente)
- **Serviços:** `PhoneService` (telefone), `MultipleLines` (várias linhas), `InternetService` (tipo de internet), `OnlineSecurity` (segurança online), `OnlineBackup` (backup online), `DeviceProtection` (proteção de dispositivo), `TechSupport` (suporte técnico), `StreamingTV`, `StreamingMovies`
- **Contrato e pagamento:** `Contract` (tipo de contrato), `PaperlessBilling` (fatura digital), `PaymentMethod` (forma de pagamento)
- **Financeiro:** `MonthlyCharges` (mensalidade), `TotalCharges` (total pago), `MonthlyIncome` (renda mensal)
- **Experiência e cancelamento:** `CustomerFeedback` (comentário do cliente), `Churn` (cancelou ou não)

# Camada Bronze

A Bronze é a porta de entrada dos dados. A regra aqui é **não alterar nada**: o arquivo é lido como texto, guardado do jeito que chegou e registrado com informações de origem. Assim, qualquer tratamento feito depois pode ser refeito ou auditado a partir desta camada.

Além de guardar, a Bronze faz o diagnóstico de qualidade. A correção dos problemas fica para a Silver.

## 1. Ingestão

In [1]:
import hashlib
import json
from datetime import datetime

import pandas as pd

CAMINHO_RAW = '../Data/Raw/history.csv'

# Tudo é lido como texto para que nenhum valor seja convertido ou perdido na leitura
df_bronze = pd.read_csv(CAMINHO_RAW, dtype=str, keep_default_na=False)

print('Linhas e colunas:', df_bronze.shape)
df_bronze.head(3)

Linhas e colunas: (5634, 23)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,CustomerFeedback,MonthlyIncome
0,4578-PHJYZ,Male,0,Yes,Yes,52,Yes,No,DSL,No,Yes,Yes,Yes,Yes,No,One year,Yes,Electronic check,68.75,3482.85,No,"I have been a customer with this internet provider for over four years now and I have been satisfied with the service. The DSL internet is reliable and I have had no issues with connectivity. The monthly charges of $68.75 are reasonable for the service I receive. I appreciate the convenience of paying with electronic check. Overall, I am happy with my experience and have no plans to switch providers.",6532
1,6289-CPNLD,Male,0,Yes,Yes,33,Yes,No,DSL,No,No,Yes,Yes,Yes,Yes,One year,Yes,Mailed check,73.9,2405.05,Yes,"I've been a customer with this company for over 2 and a half years now, and unfortunately I have decided to churn. The DSL internet service was reliable for the most part, but the monthly charges of $73.9 were a bit steep for the quality of service I was receiving. Additionally, having to mail in a check for payment was inconvenient and outdated. I ultimately decided to switch to a different provider that offered better value and more modern payment options.",7634
2,2682-KEVRP,Female,1,No,No,22,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,One year,Yes,Mailed check,20.05,417.0,No,"I have been a customer with this company for 22 months now and I have been satisfied with the service. I have not experienced any issues with billing or service interruptions. The monthly charges are affordable at $20.05 and I appreciate being able to pay by mailing a check. The lack of internet service is not a problem for me as I do not require it. Overall, I am happy with my experience as a customer and have no complaints.",3628


### Integridade do arquivo

Calculamos o hash SHA-256 do arquivo original. Ele funciona como uma impressão digital: se alguém alterar um único caractere do `history.csv`, o hash muda. Guardando esse valor junto da Bronze, dá para provar mais tarde que a análise partiu do arquivo certo.

In [2]:
with open(CAMINHO_RAW, 'rb') as arquivo:
    hash_raw = hashlib.sha256(arquivo.read()).hexdigest()

metadados = {
    'arquivo_origem': 'history.csv',
    'fonte': 'Databricks (base fictícia Terra Signal)',
    'data_ingestao': datetime.now().strftime('%Y-%m-%d %H:%M'),
    'linhas': int(df_bronze.shape[0]),
    'colunas': int(df_bronze.shape[1]),
    'sha256': hash_raw,
}
metadados

{'arquivo_origem': 'history.csv', 'fonte': 'Databricks (base fictícia Terra Signal)', 'data_ingestao': '2026-11-19 17:15', 'linhas': 5634, 'colunas': 23, 'sha256': '5898d02a7f648faab7580ef84ed13fe5a6a685e5092696174ff4b1a540d62246'}

## 2. Avaliação da qualidade dos dados

Para avaliar, usamos uma cópia em que os números são convertidos. A tabela Bronze em si continua só com texto.

In [3]:
df = pd.read_csv(CAMINHO_RAW)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5634 entries, 0 to 5633
Data columns (total 23 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        5634 non-null   str    
 1   gender            5634 non-null   str    
 2   SeniorCitizen     5634 non-null   int64  
 3   Partner           5634 non-null   str    
 4   Dependents        5634 non-null   str    
 5   tenure            5634 non-null   str    
 6   PhoneService      5634 non-null   str    
 7   MultipleLines     5634 non-null   str    
 8   InternetService   5634 non-null   str    
 9   OnlineSecurity    5634 non-null   str    
 10  OnlineBackup      5634 non-null   str    
 11  DeviceProtection  5634 non-null   str    
 12  TechSupport       5634 non-null   str    
 13  StreamingTV       5634 non-null   str    
 14  StreamingMovies   5634 non-null   str    
 15  Contract          5634 non-null   str    
 16  PaperlessBilling  5634 non-null   str    
 17  Paymen

Logo de cara aparecem dois pontos: `tenure` (tempo de assinatura em meses) foi lido como texto, o que indica algum valor não numérico na coluna, e `TotalCharges` tem valores vazios.

### 2.1 Valores ausentes

In [4]:
resumo_ausentes = pd.DataFrame({
    'Nulos': df.isna().sum(),
    'Nulos (%)': (df.isna().mean() * 100).round(2)
})
resumo_ausentes[resumo_ausentes['Nulos'] > 0]

,Nulos,Nulos (%)
TotalCharges,278,4.93


### 2.2 Duplicados e identificação

In [5]:
print('Linhas duplicadas:', df.duplicated().sum())
print('IDs de cliente repetidos:', df['customerID'].duplicated().sum())

id_valido = df['customerID'].str.fullmatch(r'[0-9]{4}-[A-Z]{5}')
print('IDs fora do padrão 0000-AAAAA:', (~id_valido).sum())

Linhas duplicadas: 0
IDs de cliente repetidos: 0
IDs fora do padrão 0000-AAAAA: 0


Não há linhas nem clientes duplicados e todos os IDs seguem o mesmo padrão. Cada linha é um cliente diferente.

### 2.3 Valores distintos das colunas de texto

É aqui que costumam aparecer erros de digitação e diferenças de maiúsculas e minúsculas.

In [6]:
colunas_texto = [c for c in df.select_dtypes(include=['object', 'string']).columns
                 if c not in ['customerID', 'CustomerFeedback', 'tenure']]

for coluna in colunas_texto:
    print(f'{coluna}: {df[coluna].value_counts(dropna=False).to_dict()}')

gender: {'Male': 2835, 'Female': 2799}
Partner: {'No': 2907, 'Yes': 2727}
Dependents: {'No': 3949, 'Yes': 1685}
PhoneService: {'Yes': 5041, 'No': 523, 'no': 40, 'yes': 30}
MultipleLines: {'No': 2704, 'Yes': 2367, 'No phone service': 563}
InternetService: {'Fiber optic': 2477, 'DSL': 1924, 'No': 1233}
OnlineSecurity: {'No': 2803, 'Yes': 1598, 'No internet service': 1233}
OnlineBackup: {'No': 2457, 'Yes': 1944, 'No internet service': 1233}
DeviceProtection: {'No': 2469, 'Yes': 1932, 'No internet service': 1233}
TechSupport: {'No': 2791, 'Yes': 1610, 'No internet service': 1233}
StreamingTV: {'No': 2219, 'Yes': 2182, 'No internet service': 1233}
StreamingMovies: {'No': 2224, 'Yes': 2177, 'No internet service': 1233}
Contract: {'Month-to-month': 3099, 'Two year': 1347, 'One year': 1188}
PaperlessBilling: {'Yes': 3327, 'No': 2307}
PaymentMethod: {'Electronic check': 1888, 'Mailed check': 1283, 'Credit card (automatic)': 1234, 'Bank transfer (automatic)': 1229}
Churn: {'No': 4130, 'Yes': 150

O único problema de escrita está em `PhoneService`, que tem "yes" e "no" em minúsculas além de "Yes" e "No". As demais colunas estão consistentes.

Em `tenure`, o valor que impede a coluna de ser numérica é a palavra "unknown":

In [7]:
tenure_numerico = pd.to_numeric(df['tenure'], errors='coerce')
df.loc[tenure_numerico.isna(), 'tenure'].value_counts()

,count
tenure,
unknown,19


### 2.4 Consistência entre colunas

Algumas colunas dependem de outras. Quem não tem telefone não pode ter várias linhas, e quem não tem internet não pode ter serviços adicionais de internet.

In [8]:
sem_telefone = df['PhoneService'].str.lower() == 'no'
print('Sem telefone, mas com MultipleLines diferente de "No phone service":',
      (sem_telefone & (df['MultipleLines'] != 'No phone service')).sum())

adicionais = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
              'TechSupport', 'StreamingTV', 'StreamingMovies']
sem_internet = df['InternetService'] == 'No'
inconsistentes = (df.loc[sem_internet, adicionais] != 'No internet service').any(axis=1).sum()
print('Sem internet, mas com algum serviço adicional:', inconsistentes)

Sem telefone, mas com MultipleLines diferente de "No phone service": 0
Sem internet, mas com algum serviço adicional: 0


As regras são respeitadas em todas as linhas.

### 2.5 Faixas dos valores numéricos

In [9]:
df[['MonthlyCharges', 'TotalCharges', 'MonthlyIncome']].describe().round(2)

,MonthlyCharges,TotalCharges,MonthlyIncome
count,5634.00,5356.00,5634.00
mean,64.61,2271.88,5104.88
std,30.15,2253.78,2839.23
min,18.40,18.80,100.00
25%,35.16,412.90,2693.75
50%,70.30,1390.72,5079.50
75%,89.80,3729.75,7574.75
max,118.65,8684.80,9993.00


Não há valores negativos nem absurdos. A mensalidade vai de 18 a 119 e a renda de 100 a 9.993.

### 2.6 Relação entre total pago, tempo e mensalidade

O valor total pago deveria ser parecido com o tempo de assinatura multiplicado pela mensalidade. Se essa relação se confirmar, ela serve para tratar os valores ausentes na Silver.

In [10]:
validos = df['TotalCharges'].notna() & (tenure_numerico > 0)
razao = df.loc[validos, 'TotalCharges'] / (tenure_numerico[validos] * df.loc[validos, 'MonthlyCharges'])
razao.describe().round(3)

,0
count,5338.000
mean,1.000
std,0.051
min,0.689
25%,0.980
50%,1.000
75%,1.020
max,1.495


A mediana da razão é 1,00 e metade dos clientes fica entre 0,98 e 1,02. Ou seja, `TotalCharges` é praticamente `tenure × MonthlyCharges`; as pequenas diferenças vêm de reajustes de mensalidade ao longo do tempo.

In [11]:
total_vazio = df['TotalCharges'].isna()
print('Clientes com TotalCharges vazio:', total_vazio.sum())
print('Desses, com tenure = 0:', (tenure_numerico[total_vazio] == 0).sum())
print('Clientes com tenure = 0 na base inteira:', (tenure_numerico == 0).sum())
print('Clientes com tenure e TotalCharges vazios ao mesmo tempo:', (total_vazio & tenure_numerico.isna()).sum())

Clientes com TotalCharges vazio: 278
Desses, com tenure = 0: 10
Clientes com tenure = 0 na base inteira: 10
Clientes com tenure e TotalCharges vazios ao mesmo tempo: 1


Os 10 clientes com `tenure = 0` acabaram de entrar e ainda não pagaram nenhuma fatura, por isso o total é vazio. Os outros 268 vazios estão espalhados pela base, sem relação com o tempo de casa.

### 2.7 Dados sensíveis

A base não tem nome, CPF, endereço, telefone ou e-mail. Os pontos que pedem cuidado são:

- `customerID`: identifica um cliente específico. Fica disponível até a Silver e é substituído por um código anônimo nas Golds.
- `CustomerFeedback`: texto livre escrito pelo cliente, que pode conter informação pessoal. Fica na Silver e não vai para as Golds.
- `MonthlyIncome`: informação financeira pessoal. Só é usada de forma agregada (médias e faixas).

## 3. Diagnóstico entregue para a Silver

| Ponto encontrado | Quantidade | Tratamento previsto na Silver |
|---|---|---|
| `PhoneService` com "yes"/"no" minúsculos | 70 | Padronizar a escrita |
| `tenure` com "unknown" | 19 | Estimar pelo total pago ÷ mensalidade |
| `TotalCharges` vazio com `tenure = 0` | 10 | Preencher com 0 (cliente novo) |
| `TotalCharges` vazio nos demais | 268 | Estimar por `tenure × MonthlyCharges` |
| Cliente com `tenure` e `TotalCharges` vazios | 1 | Remover (não há como estimar) |
| Nomes e valores em inglês | todas | Traduzir para português |
| Tipos errados (texto em vez de número/booleano) | várias | Corrigir a tipagem |
| Duplicados, IDs inválidos, inconsistências entre colunas | 0 | Nenhum |

## 4. Gravação da camada Bronze

A tabela é salva exatamente como foi lida, junto com o arquivo de metadados.

In [12]:
df_bronze.to_csv('../Data/Bronze/tabela_bronze.csv', index=False, encoding='utf-8-sig')

with open('../Data/Bronze/metadados_ingestao.json', 'w', encoding='utf-8') as arquivo:
    json.dump(metadados, arquivo, ensure_ascii=False, indent=2)

# Conferência: a Bronze salva tem o mesmo conteúdo do arquivo original
conferencia = pd.read_csv('../Data/Bronze/tabela_bronze.csv', dtype=str, keep_default_na=False)
print('Bronze idêntica ao original:', conferencia.equals(df_bronze))

Bronze idêntica ao original: True
